# 04 - Simulation Truth And Lateral Distance

Use GCD geometry and simulation truth to compute the perpendicular distance between pulsed in-ice DOMs and a truth-track candidate.


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from icetray_tutorial.paths import SIMULATION
from icetray_tutorial.frames import event_header_dict, iter_frames, stop_name
from icetray_tutorial.geometry import inice_dom_positions, pulsed_dom_lateral_distances, read_geometry
from icetray_tutorial.pulses import summarize_pulses

In [ ]:
geometry = read_geometry(SIMULATION.gcd)
positions = inice_dom_positions(geometry)
len(positions)

In [ ]:
xy = pd.DataFrame([{'string': omkey.string, 'om': omkey.om, 'x': pos[0], 'y': pos[1], 'z': pos[2]} for omkey, pos in positions.items()])
ax = xy.plot.scatter(x='x', y='y', s=4, alpha=0.4, figsize=(6, 6))
ax.set_aspect('equal')
ax.set_title('InIce DOM positions')

In [ ]:
rows = []
physics_seen = 0
for frame in iter_frames(SIMULATION.event_file):
    if stop_name(frame) != 'Physics':
        continue
    physics_seen += 1
    pulse_summary = summarize_pulses(frame)
    if pulse_summary is None or pulse_summary.hit_doms < 10:
        continue
    distances = pulsed_dom_lateral_distances(frame, geometry)
    if distances:
        header = event_header_dict(frame)
        for row in distances:
            rows.append({**header, **row, 'event_hit_doms': pulse_summary.hit_doms})
    if physics_seen >= 200:
        break

distance_table = pd.DataFrame(rows)
distance_table.head()

In [ ]:
distance_table['distance_m'].dropna().plot.hist(bins=60)
plt.xlabel('DOM lateral distance to truth track [m]')
plt.ylabel('pulsed DOMs')

## Practice

1. Inspect keys containing `MC`, `Muon`, or `Primary`.
2. Try a stricter hit DOM requirement.
3. Choose a better truth key if the frame provides one.
